# Lizard Maintainability — Nesting Depth Raw Output Extraction (C)

This notebook analyzes **C language repositories** with **Lizard** and captures the complete raw tool output for maintainability nesting-depth metric derivation and validation.

**Default benchmark repository:** [redis/redis](https://github.com/redis/redis)

The notebook supports:
- **Mode 1:** Clone from a Git repository URL
- **Mode 2:** Analyze an already-cloned local repository path

All deliverables are written to the configured `OUTPUT_DIR`.

## Section 1 — Install Dependencies

Install open-source packages required for repository acquisition, static analysis, and result processing.

In [ ]:
!pip install -q lizard pandas gitpython jupyter

## Section 2 — Configuration

Set execution mode, repository source, output location, and Lizard options.

- Set `USE_GIT_URL = True` to clone from `REPO_URL`.
- Set `USE_GIT_URL = False` to analyze `LOCAL_REPO_PATH` directly.
- When cloning, use `IF_CLONE_EXISTS` to choose between reusing or re-cloning an existing local copy.

In [ ]:
USE_GIT_URL = True

REPO_URL = "https://github.com/redis/redis.git"

LOCAL_REPO_PATH = "/content/redis"

# Fast validation benchmark (100% predictable outcomes):
# LOCAL_REPO_PATH = "./workspace/c_nesting_benchmark"

OUTPUT_DIR = "./outputs"

# Clone behavior when USE_GIT_URL is True and the target directory already exists.
# Options: "reuse" (keep existing clone) or "reclone" (delete and clone again)
IF_CLONE_EXISTS = "reuse"

# Shallow clone depth for Git URL mode (None = full clone)
CLONE_DEPTH = 1

WORKSPACE_DIR = "./workspace"

# Lizard language flag for C/C++ sources
LIZARD_LANGUAGE = "cpp"

# Use Lizard -ENS extension for nested control structure counting
USE_NESTING_EXTENSION = True

# Stream raw Lizard stdout/stderr to the notebook console during execution
STREAM_RAW_OUTPUT = True

# Number of raw output lines to preview after execution
RAW_OUTPUT_PREVIEW_LINES = 150

## Utility Functions

Modular helpers for logging, repository setup, C file discovery, Lizard execution, and nesting-depth extraction.

In [ ]:
from __future__ import annotations

import csv
import io
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError


EXCLUDED_DIR_NAMES = {
    ".git",
    "build",
    "dist",
    "out",
    "bin",
    "obj",
    "vendor",
    "third_party",
    "tests",
    "docs",
}

C_FILE_EXTENSIONS = {".c", ".h"}

LIZARD_EXCLUDE_PATTERNS = [
    "*/.git/*",
    "*/build/*",
    "*/dist/*",
    "*/out/*",
    "*/bin/*",
    "*/obj/*",
    "*/vendor/*",
    "*/third_party/*",
    "*/tests/*",
    "*/docs/*",
]

LIZARD_CSV_COLUMNS = [
    "nloc",
    "ccn",
    "token_count",
    "parameter_count",
    "length",
    "location",
    "file",
    "function",
    "long_name",
    "start_line",
    "end_line",
]

LIZARD_CSV_COLUMNS_ENS = LIZARD_CSV_COLUMNS + ["max_nested_structures"]


class NotebookLogger:
    """Append-only logger for console progress and persistent error logging."""

    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.error_log_path.exists():
            self.error_log_path.write_text("", encoding="utf-8")

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        with self.error_log_path.open("a", encoding="utf-8") as handle:
            handle.write(line)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)

    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')

    logger.info(f"Cloning {repo_url} into {clone_path} (depth={clone_depth})")
    try:
        clone_kwargs = {"depth": clone_depth} if clone_depth else {}
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Clone failed for {repo_url}: {exc}")
        raise

    logger.info(f"Clone completed: {clone_path}")
    return clone_path.resolve()


def validate_local_repository(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        message = f"Local repository path does not exist: {local_repo_path}"
        logger.error(message)
        raise FileNotFoundError(message)

    if not local_repo_path.is_dir():
        message = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(message)
        raise NotADirectoryError(message)

    has_git = (local_repo_path / ".git").exists()
    has_c_source = any(
        file_path.suffix.lower() in C_FILE_EXTENSIONS
        for file_path in local_repo_path.rglob("*")
        if file_path.is_file()
    )

    if not has_git and not has_c_source:
        message = (
            f"Path is neither a Git repository nor a C source directory: {local_repo_path}"
        )
        logger.error(message)
        raise ValueError(message)

    if has_git:
        try:
            Repo(local_repo_path)
        except InvalidGitRepositoryError as exc:
            logger.error(f"Invalid Git repository at {local_repo_path}: {exc}")
            raise

    logger.info(f"Validated local repository path: {local_repo_path}")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool,
    repo_url: str,
    local_repo_path: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        logger.info("Execution mode: Git repository URL")
        return clone_or_reuse_repository(
            repo_url, workspace_dir, if_clone_exists, logger, clone_depth=clone_depth
        )

    logger.info("Execution mode: Local repository path")
    return validate_local_repository(Path(local_repo_path), logger)


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def discover_c_files(repo_path: Path) -> list[Path]:
    c_files: list[Path] = []
    for file_path in repo_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in C_FILE_EXTENSIONS:
            continue
        if should_exclude_path(file_path.relative_to(repo_path)):
            continue
        c_files.append(file_path.resolve())
    c_files.sort()
    return c_files


def compute_repository_stats(repo_path: Path, c_files: list[Path]) -> dict[str, Any]:
    total_size_bytes = sum(file_path.stat().st_size for file_path in c_files)
    directory_count = sum(
        1
        for current_path, _, _ in os.walk(repo_path)
        if not should_exclude_path(Path(current_path).relative_to(repo_path))
    )
    return {
        "c_file_count": len(c_files),
        "repository_size_bytes": total_size_bytes,
        "directory_count": directory_count,
    }


def save_c_file_list(c_files: list[Path], repo_path: Path, output_csv: Path) -> None:
    rows = [
        {
            "absolute_path": str(file_path),
            "relative_path": str(file_path.relative_to(repo_path)),
            "extension": file_path.suffix.lower(),
        }
        for file_path in c_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def build_lizard_command(
    repo_path: Path,
    language: str,
    output_csv: bool = False,
    output_xml: bool = False,
    use_nesting_extension: bool = False,
) -> list[str]:
    command = [sys.executable, "-m", "lizard", "-l", language]
    for pattern in LIZARD_EXCLUDE_PATTERNS:
        command.extend(["-x", pattern])
    if output_csv:
        command.append("--csv")
    if output_xml:
        command.append("-X")
    if use_nesting_extension:
        command.append("-ENS")
    command.append(str(repo_path))
    return command


def run_lizard_command(
    command: list[str],
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> tuple[str, str, int, bool]:
    try:
        if stream_raw:
            process = subprocess.Popen(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace",
                env=os.environ.copy(),
            )
            stdout_lines: list[str] = []
            stderr_lines: list[str] = []

            def _read_stream(pipe, sink, label):
                for line in iter(pipe.readline, ""):
                    print(line, end="", file=sys.stderr if label == "stderr" else sys.stdout)
                    sink.append(line)
                pipe.close()

            assert process.stdout is not None
            assert process.stderr is not None
            stdout_thread = threading.Thread(
                target=_read_stream, args=(process.stdout, stdout_lines, "stdout"), daemon=True
            )
            stderr_thread = threading.Thread(
                target=_read_stream, args=(process.stderr, stderr_lines, "stderr"), daemon=True
            )
            stdout_thread.start()
            stderr_thread.start()
            return_code = process.wait()
            stdout_thread.join()
            stderr_thread.join()
            stdout = "".join(stdout_lines)
            stderr = "".join(stderr_lines)
        else:
            completed = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                check=False,
                env=os.environ.copy(),
            )
            stdout = completed.stdout
            stderr = completed.stderr
            return_code = completed.returncode

        success = return_code == 0
        if not success:
            logger.error(f"Lizard command failed with exit code {return_code}: {' '.join(command)}")
        return stdout, stderr, return_code, success
    except Exception as exc:
        logger.error(f"Lizard execution exception: {exc}")
        return "", str(exc), -1, False


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw_output = stdout
    if stderr:
        if raw_output and not raw_output.endswith("\n"):
            raw_output += "\n"
        raw_output += stderr
    return raw_output


def run_lizard_suite(
    repo_path: Path,
    language: str,
    logger: NotebookLogger,
    stream_raw: bool = False,
    use_nesting_extension: bool = True,
) -> dict[str, str]:
    logger.info(f"Running Lizard on repository: {repo_path}")

    raw_command = build_lizard_command(repo_path, language)
    raw_stdout, raw_stderr, _, raw_ok = run_lizard_command(raw_command, logger, stream_raw=stream_raw)

    csv_command = build_lizard_command(repo_path, language, output_csv=True)
    csv_stdout, csv_stderr, _, csv_ok = run_lizard_command(csv_command, logger, stream_raw=False)
    if csv_stderr.strip():
        logger.error(f"Lizard CSV stderr: {csv_stderr.strip()}")

    xml_command = build_lizard_command(repo_path, language, output_xml=True)
    xml_stdout, xml_stderr, _, xml_ok = run_lizard_command(xml_command, logger, stream_raw=False)
    if xml_stderr.strip():
        logger.error(f"Lizard XML stderr: {xml_stderr.strip()}")

    ens_csv_stdout = ""
    if use_nesting_extension:
        ens_command = build_lizard_command(
            repo_path, language, output_csv=True, use_nesting_extension=True
        )
        ens_csv_stdout, ens_stderr, _, ens_ok = run_lizard_command(ens_command, logger, stream_raw=False)
        if ens_stderr.strip():
            logger.error(f"Lizard ENS CSV stderr: {ens_stderr.strip()}")
        if not ens_ok:
            logger.error("Lizard ENS CSV run failed; nesting depth derivation fallback will be used.")

    return {
        "raw_text": combine_raw_streams(raw_stdout, raw_stderr),
        "csv_text": csv_stdout,
        "xml_text": xml_stdout,
        "ens_csv_text": ens_csv_stdout,
        "execution_ok": str(raw_ok or csv_ok or xml_ok),
    }


def parse_lizard_csv(csv_text: str, with_ens: bool = False) -> pd.DataFrame:
    if not csv_text.strip():
        return pd.DataFrame(columns=LIZARD_CSV_COLUMNS_ENS if with_ens else LIZARD_CSV_COLUMNS)

    columns = LIZARD_CSV_COLUMNS_ENS if with_ens else LIZARD_CSV_COLUMNS
    reader = csv.reader(io.StringIO(csv_text.strip()))
    rows = list(reader)
    if not rows:
        return pd.DataFrame(columns=columns)

    if rows[0] and rows[0][0].lower() in {"nloc", "ncss"}:
        rows = rows[1:]

    parsed_rows = []
    for row in rows:
        if len(row) < len(columns):
            row = row + [""] * (len(columns) - len(row))
        parsed_rows.append(dict(zip(columns, row[: len(columns)])))

    frame = pd.DataFrame(parsed_rows)
    for col in ["nloc", "ccn", "token_count", "parameter_count", "length", "start_line", "end_line"]:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    if with_ens and "max_nested_structures" in frame.columns:
        frame["max_nested_structures"] = pd.to_numeric(
            frame["max_nested_structures"], errors="coerce"
        )
    return frame


def strip_c_comments_and_strings(line: str) -> str:
    line = re.sub(r"//.*$", "", line)
    line = re.sub(r"/\*.*?\*/", "", line)
    line = re.sub(r'"(\\.|[^"\\])*"', '""', line)
    line = re.sub(r"'(\\.|[^'\\])*'", "''", line)
    return line


def derive_max_nesting_depth_from_source(source_text: str, start_line: int, end_line: int) -> int:
    lines = source_text.splitlines()
    if start_line < 1 or end_line > len(lines):
        return 0

    body_lines = lines[start_line - 1 : end_line]
    max_depth = 0
    brace_depth = 0
    control_pattern = re.compile(r"\b(if|for|while|switch|do)\b")

    for line in body_lines:
        code = strip_c_comments_and_strings(line)
        if control_pattern.search(code):
            max_depth = max(max_depth, brace_depth + 1)
        for char in code:
            if char == "{":
                brace_depth += 1
                max_depth = max(max_depth, brace_depth)
            elif char == "}":
                brace_depth = max(brace_depth - 1, 0)

    return int(max_depth)


def build_nesting_depth_results(
    csv_df: pd.DataFrame,
    ens_csv_df: pd.DataFrame,
    repo_path: Path,
    logger: NotebookLogger,
) -> tuple[pd.DataFrame, int]:
    if csv_df.empty and ens_csv_df.empty:
        return pd.DataFrame(
            columns=["file", "function", "start_line", "end_line", "max_nesting_depth", "status"]
        ), 0

    base_df = ens_csv_df if not ens_csv_df.empty else csv_df
    rows: list[dict[str, Any]] = []
    files_failed = 0
    source_cache: dict[str, str] = {}

    for _, record in base_df.iterrows():
        file_path = Path(str(record.get("file", "")).strip('"'))
        function_name = str(record.get("function", "")).strip('"')
        start_line = int(record.get("start_line", 0) or 0)
        end_line = int(record.get("end_line", 0) or 0)

        reported_depth = None
        if not ens_csv_df.empty and function_name:
            match = ens_csv_df[
                (ens_csv_df["function"].astype(str).str.strip('"') == function_name)
                & (ens_csv_df["start_line"] == start_line)
            ]
            if not match.empty and "max_nested_structures" in match.columns:
                reported_depth = int(match.iloc[0]["max_nested_structures"])

        if reported_depth is not None:
            rows.append(
                {
                    "file": str(file_path),
                    "function": function_name,
                    "start_line": start_line,
                    "end_line": end_line,
                    "max_nesting_depth": reported_depth,
                    "status": "reported",
                }
            )
            continue

        try:
            file_key = str(file_path)
            if file_key not in source_cache:
                source_cache[file_key] = Path(file_key).read_text(encoding="utf-8", errors="replace")
            derived_depth = derive_max_nesting_depth_from_source(
                source_cache[file_key], start_line, end_line
            )
            rows.append(
                {
                    "file": str(file_path),
                    "function": function_name,
                    "start_line": start_line,
                    "end_line": end_line,
                    "max_nesting_depth": derived_depth,
                    "status": "derived",
                }
            )
        except Exception as exc:
            files_failed += 1
            logger.error(f"Failed to derive nesting depth for {file_path}::{function_name}: {exc}")
            rows.append(
                {
                    "file": str(file_path),
                    "function": function_name,
                    "start_line": start_line,
                    "end_line": end_line,
                    "max_nesting_depth": None,
                    "status": "failed",
                }
            )

    return pd.DataFrame(rows), files_failed


def compute_maintainability_summary(nesting_df: pd.DataFrame) -> pd.DataFrame:
    valid = nesting_df[nesting_df["max_nesting_depth"].notna()].copy()
    if valid.empty:
        return pd.DataFrame(
            [
                {"metric_name": "Maintainability_Nesting_Depth", "metric_value": 0},
                {"metric_name": "Average_Nesting_Depth", "metric_value": 0},
            ]
        )

    max_depth = int(valid["max_nesting_depth"].max())
    avg_depth = float(valid["max_nesting_depth"].mean())
    return pd.DataFrame(
        [
            {"metric_name": "Maintainability_Nesting_Depth", "metric_value": max_depth},
            {"metric_name": "Average_Nesting_Depth", "metric_value": round(avg_depth, 4)},
        ]
    )


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW LIZARD OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")

## Section 3 — Repository Setup

Resolve the repository path based on configuration, initialize output directories, and prepare for Lizard execution.

In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / "error_log.txt"

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f"Repository setup failed: {exc}")
    raise

logger.info(f"Repository ready at: {REPO_PATH}")

## Section 4 — Discover C Files

Recursively discover `.c` and `.h` files while excluding common non-source directories.

In [ ]:
C_FILES = discover_c_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, C_FILES)

C_FILES_CSV = OUTPUT_PATH / "c_files.csv"
save_c_file_list(C_FILES, REPO_PATH, C_FILES_CSV)

print(f"Total C Files Found: {len(C_FILES)}")
print(f"Repository Size (C files only): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Total Directories (excluding filtered paths): {REPO_STATS['directory_count']:,}")
print(f"Saved file list to: {C_FILES_CSV}")

## Section 5 — Execute Lizard

Run Lizard against the repository and capture complete raw output.

- Default analysis: `lizard <repo_path>`
- Additional runs for CSV/XML are performed in Section 6
- `-ENS` extension is used internally for nesting-depth extraction in Section 7
- Raw stdout/stderr are preserved without suppression

In [ ]:
if not C_FILES:
    logger.error("No C files discovered; skipping Lizard execution.")
    LIZARD_OUTPUTS = {
        "raw_text": "",
        "csv_text": "",
        "xml_text": "",
        "ens_csv_text": "",
        "execution_ok": "False",
    }
else:
    LIZARD_OUTPUTS = run_lizard_suite(
        repo_path=REPO_PATH,
        language=LIZARD_LANGUAGE,
        logger=logger,
        stream_raw=STREAM_RAW_OUTPUT,
        use_nesting_extension=USE_NESTING_EXTENSION,
    )

logger.info(f"Lizard execution complete. Raw output size: {len(LIZARD_OUTPUTS['raw_text']):,} characters")

## Section 6 — Raw Output Extraction

Persist complete raw Lizard text output, CSV output, and XML output exactly as emitted by the tool.

In [ ]:
RAW_OUTPUT_PATH = OUTPUT_PATH / "lizard_raw_output.txt"
CSV_OUTPUT_PATH = OUTPUT_PATH / "lizard_output.csv"
XML_OUTPUT_PATH = OUTPUT_PATH / "lizard_output.xml"

raw_text_output = LIZARD_OUTPUTS["raw_text"]
RAW_OUTPUT_PATH.write_text(raw_text_output, encoding="utf-8")
CSV_OUTPUT_PATH.write_text(LIZARD_OUTPUTS["csv_text"], encoding="utf-8")
XML_OUTPUT_PATH.write_text(LIZARD_OUTPUTS["xml_text"], encoding="utf-8")

LIZARD_CSV_DF = parse_lizard_csv(LIZARD_OUTPUTS["csv_text"], with_ens=False)
LIZARD_ENS_CSV_DF = parse_lizard_csv(LIZARD_OUTPUTS["ens_csv_text"], with_ens=True)

logger.info(f"Saved raw output: {RAW_OUTPUT_PATH}")
logger.info(f"Saved CSV output: {CSV_OUTPUT_PATH}")
logger.info(f"Saved XML output: {XML_OUTPUT_PATH}")
logger.info(f"Functions parsed from CSV: {len(LIZARD_CSV_DF)}")

preview_raw_output(raw_text_output, RAW_OUTPUT_PREVIEW_LINES, RAW_OUTPUT_PATH)

## Section 7 — Maintainability Nesting Depth Extraction

Extract function-level nesting depth using Lizard `-ENS` nested control structure metrics.

If nesting depth is unavailable from Lizard output, derive it by parsing the function control-structure hierarchy from source code.

In [ ]:
NESTING_RESULTS_DF, FILES_FAILED = build_nesting_depth_results(
    csv_df=LIZARD_CSV_DF,
    ens_csv_df=LIZARD_ENS_CSV_DF,
    repo_path=REPO_PATH,
    logger=logger,
)

NESTING_RESULTS_CSV = OUTPUT_PATH / "nesting_depth_results.csv"
NESTING_RESULTS_DF.to_csv(NESTING_RESULTS_CSV, index=False)

logger.info(f"Saved nesting depth results: {NESTING_RESULTS_CSV}")
logger.info(f"Functions analyzed: {len(NESTING_RESULTS_DF)}")

if not NESTING_RESULTS_DF.empty:
    display(NESTING_RESULTS_DF.head(10))
else:
    print("No functions found for nesting depth extraction.")

## Section 8 — Metric Computation

Compute repository-level maintainability nesting depth metrics:

- **Maintainability_Nesting_Depth** = `max(max_nesting_depth(function_i))`
- **Average_Nesting_Depth** = `Σ max_nesting_depth(function_i) / total_functions`

In [ ]:
SUMMARY_DF = compute_maintainability_summary(NESTING_RESULTS_DF)
SUMMARY_CSV = OUTPUT_PATH / "maintainability_nesting_depth_summary.csv"
SUMMARY_DF.to_csv(SUMMARY_CSV, index=False)

logger.info(f"Saved maintainability summary: {SUMMARY_CSV}")
display(SUMMARY_DF)

## Section 9 — Summary Dashboard

Overview of analysis coverage and nesting-depth metrics.

In [ ]:
max_depth = SUMMARY_DF.loc[
    SUMMARY_DF["metric_name"] == "Maintainability_Nesting_Depth", "metric_value"
].iloc[0]
avg_depth = SUMMARY_DF.loc[
    SUMMARY_DF["metric_name"] == "Average_Nesting_Depth", "metric_value"
].iloc[0]

summary_df = pd.DataFrame(
    [
        {"Metric": "Total C Files", "Value": len(C_FILES)},
        {"Metric": "Total Functions", "Value": len(LIZARD_CSV_DF)},
        {"Metric": "Functions Analyzed", "Value": len(NESTING_RESULTS_DF)},
        {"Metric": "Maximum Nesting Depth", "Value": max_depth},
        {"Metric": "Average Nesting Depth", "Value": avg_depth},
        {"Metric": "Files Failed", "Value": FILES_FAILED},
    ]
)

display(summary_df)

deliverables = [
    RAW_OUTPUT_PATH,
    CSV_OUTPUT_PATH,
    XML_OUTPUT_PATH,
    C_FILES_CSV,
    NESTING_RESULTS_CSV,
    SUMMARY_CSV,
    ERROR_LOG_PATH,
]

print("\nDeliverables:")
for deliverable in deliverables:
    status = "OK" if deliverable.exists() else "MISSING"
    print(f"  [{status}] {deliverable}")

## Section 10 — Error Handling

Failures encountered during cloning, validation, or Lizard execution are appended to `outputs/error_log.txt`.

In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding="utf-8"))
else:
    print("No errors logged.")

## Section 11 — Deliverables

Upon successful completion, the following artifacts are available under `outputs/`:

```text
outputs/
├── lizard_raw_output.txt
├── lizard_output.csv
├── lizard_output.xml
├── c_files.csv
├── nesting_depth_results.csv
├── maintainability_nesting_depth_summary.csv
└── error_log.txt
```

The notebook is designed to run end-to-end in Jupyter Notebook and Google Colab without manual intervention.